# MTech Bug-Fix Agent: Colab Feasibility Check

This short notebook checks the free Colab runtime and installs the project from GitHub. We will use the GPU and memory details to choose a model that fits. It does not download a model or run benchmark tasks yet.

## Before running

In Colab, choose **Runtime → Change runtime type** and select a GPU if one is available. Then run the cells from top to bottom. Free GPU availability can vary.

In [16]:
import platform
import sys
import torch

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print('GPU memory (GiB):', round(props.total_memory / (1024 ** 3), 2))
else:
    print('GPU: none selected or currently available')
print('Python executable:', sys.executable)

Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GiB): 14.56
Python executable: /usr/bin/python3


## Download and install the project

The repository is public. This cell clones it into Colab's temporary storage and installs the lightweight project package. Colab storage resets when its runtime is discarded, so rerun this notebook after a reset.

In [17]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/pisalaarti05/mtech-bug-fix-agent.git'
PROJECT_DIR = Path('/content/mtech-bug-fix-agent')

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--editable', str(PROJECT_DIR)], check=True)
print('Project installed from:', PROJECT_DIR)

Project installed from: /content/mtech-bug-fix-agent


## Confirm the command is ready

This displays the program's options. A successful result means Colab can import the project. Do not choose a model yet; first share the runtime details above.

In [18]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'bugfix_agent.cli', '--help'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'bugfix_agent.cli', '--help'], returncode=0)

## Pilot the recommended code model

The candidate is Qwen2.5-Coder-7B-Instruct. Its full model files are about 15.2 GB, so the download may take a while. The next cells install the 4-bit loading tools, download the model into temporary Colab storage, and generate one short answer. The model files are not added to this Git repository. If Colab disconnects, rerun the setup and model cells.

In [19]:
import subprocess
import sys

# Install only the model libraries. Keep Colab's existing PyTorch/CUDA build.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
    'transformers', 'accelerate', 'bitsandbytes'
], check=True)
from importlib.metadata import version
for package in ('transformers', 'accelerate', 'bitsandbytes'):
    print(f'{package}:', version(package))
print('Model libraries installed.')

transformers: 5.22.1
accelerate: 1.15.0
bitsandbytes: 0.50.2
Model libraries installed.


## Load the model and make one short request

This uses 4-bit NF4 compression to reduce GPU memory use. The T4 supports this setup with the current Colab runtime. The generated text is only a format check; it is not an experiment result.

In [20]:
from pathlib import Path
import sys
import torch

PROJECT_DIR = Path('/content/mtech-bug-fix-agent')
SOURCE_DIR = PROJECT_DIR / 'src'
if not (SOURCE_DIR / 'bugfix_agent' / 'model.py').is_file():
    raise FileNotFoundError('Run the Download and install the project cell first.')
sys.path.insert(0, str(SOURCE_DIR))
from bugfix_agent.model import HuggingFaceModel

MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
torch.cuda.reset_peak_memory_stats()
model = HuggingFaceModel(MODEL_ID, max_new_tokens=128, max_input_tokens=1024)
print('Model loaded:', MODEL_ID)
print('Model snapshot:', model.revision)
print('Load time (seconds):', model.load_seconds)
print('GPU memory after load (GiB):', round(torch.cuda.memory_allocated() / (1024 ** 3), 2))

prompt = (
    'Return only a unified diff for this small Python bug. '
    'Current file: def divide(a, b): return a / b. '
    'Bug: when b is zero, return None instead of raising ZeroDivisionError.'
)
response = model.generate(prompt)
print('Generated response:')
print(response)
print('Token and time record:', model.last_usage)
print('Peak GPU memory (GiB):', round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2))
if not response.strip():
    raise RuntimeError('The model loaded but returned an empty response.')

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-Coder-7B-Instruct
Model snapshot: None
Load time (seconds): 327.683
GPU memory after load (GiB): 5.18
Generated response:
```diff
diff --git a/divide.py b/divide.py
--- a/divide.py
+++ b/divide.py
@@ -1,3 +1,4 @@
 def divide(a, b):
-    return a / b
+    if b == 0:
+        return None
+    else:
+        return a / b
```
Token and time record: {'input_tokens': 70, 'output_tokens': 76, 'generation_seconds': 5.64}
Peak GPU memory (GiB): 5.33
